# A7 Part 2, Part 2: Retrieval-Augmented Generation (RAG) Using Our SLM
**INFO 490 MG, Team 2 (Unleased)**

This notebook asks one question: *if we hand the same trained Small Language Model some extra information at question time (RAG), do its answers change?* No weights are updated at any point; we prove that at the end with a checksum of the weights taken before and after.

**The SLM is the one selected in Part 1:** Experiment 5 (embedding 96, 4 heads, 2 layers, feed-forward 384, context 32, 232,155 parameters, 30,000 steps, seed 42). The model class, tokenizer, corpus, training loop and `generate_text` below are copied from the Part 1 notebook unchanged. We do not use any pretrained or Hugging Face model.

**Two facts about this SLM shape everything below**
1. It is a *character-level* model whose vocabulary is only the characters in the class corpus: newline, space, `.` and the letters a-z **except `q` and `z`**. Anything else (digits, capitals, `?`, `,`, `:`) cannot even be encoded, so our knowledge base, questions and prompt labels are written inside that alphabet.
2. Its context window is **32 characters**. A retrieved document is longer than that, so most of what we retrieve cannot be seen by the model. We measure this instead of hiding it.

**How to run:** `python -m jupyter nbconvert --to notebook --execute --inplace a7_rag_with_slm.ipynb` (or Run all in Jupyter/Colab). The first run retrains the selected SLM on the CPU (about 5 to 10 minutes) and saves it to `slm_results/selected_slm.pt` (git-ignored). If you already have the Part 1 `selected_slm.pt`, put it in `slm_results/` and the notebook will load it instead of retraining.


## Setup

In [1]:
import math, random, time, platform, json, os, hashlib, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)           # same as Part 1
DEVICE = torch.device("cpu")       # same as Part 1: CPU only
OUT = Path("slm_results"); OUT.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", None, "display.width", 250)

print("PyTorch:", torch.__version__, "| Device:", DEVICE, "| Machine:", platform.platform())

PyTorch: 2.14.1 | Device: cpu | Machine: macOS-26.6.2-arm64-arm-64bit-Mach-O


## The selected SLM, rebuilt exactly as in Part 1 (class Cells 2 to 12A, unchanged)

In [2]:
# Architecture of the SLM selected in Part 1 (Experiment 5), see llm-test/a7-part2/selected_slm.json
BATCH_SIZE = 16
CONTEXT_LENGTH = 32
EMBEDDING_DIM = 96
NUM_HEADS = 4
NUM_LAYERS = 2
FEEDFORWARD_DIM = 384
LEARNING_RATE = 0.001
TRAINING_STEPS = 30000

p1 = Path("../a7-part2/selected_slm.json")
if p1.exists():                    # make sure we really match what Part 1 selected
    sel = json.load(open(p1))["config"]
    assert (sel["embedding_dim"], sel["num_heads"], sel["num_layers"], sel["feedforward_dim"]) == \
           (EMBEDDING_DIM, NUM_HEADS, NUM_LAYERS, FEEDFORWARD_DIM)
    print("Matches Part 1 selected_slm.json:", sel)

# ---- Training corpus (class Cell 3): 20 sentences repeated 100 times ----
sentences = [
    "students build django applications.", "students learn python programming.",
    "students develop artificial intelligence systems.", "language models predict the next token.",
    "transformers use attention mechanisms.", "neural networks learn from training data.",
    "machine learning models contain parameters.", "the professor teaches artificial intelligence.",
    "students create innovative startup applications.", "django applications use models and views.",
    "training changes the weights of a neural network.", "inference uses previously learned parameters.",
    "the university offers courses in informatics.", "students experiment with language models.",
    "artificial intelligence can generate text.", "the model predicts characters using probabilities.",
    "attention helps models use contextual information.", "python is useful for machine learning research.",
    "the students are building useful software products.", "small language models can run on local computers."]
text = ("\n".join(sentences) + "\n") * 100
CORPUS_SENTENCES = set(sentences)
CORPUS_WORDS = {w for w in re.findall(r"[a-z]+", text)}

# ---- Character-level tokenizer (class Cell 4) ----
chars = sorted(set(text))
VOCAB_SIZE = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
def encode(s): return [stoi[ch] for ch in s]
def decode(ids): return "".join(itos[int(i)] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]

def get_batch(source):
    starts = torch.randint(0, len(source) - CONTEXT_LENGTH, (BATCH_SIZE,))
    x = torch.stack([source[i:i + CONTEXT_LENGTH] for i in starts])
    y = torch.stack([source[i + 1:i + CONTEXT_LENGTH + 1] for i in starts])
    return x.to(DEVICE), y.to(DEVICE)

print("Characters in corpus:", len(text), "(Part 1: 90,300)")
print("Vocabulary size:", VOCAB_SIZE, "->", repr("".join(chars)))
print("Letters the model has never seen:", [c for c in "abcdefghijklmnopqrstuvwxyz" if c not in stoi])

# ---- Transformer (class Cells 9A and 9B) ----
class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, feedforward_dim):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=num_heads, dropout=0.0, batch_first=True)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.feedforward = nn.Sequential(nn.Linear(embedding_dim, feedforward_dim), nn.ReLU(),
                                         nn.Linear(feedforward_dim, embedding_dim))
    def forward(self, x):
        T = x.shape[1]
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        normalized = self.ln1(x)
        attended, _ = self.attention(normalized, normalized, normalized, attn_mask=mask, need_weights=False)
        x = x + attended
        x = x + self.feedforward(self.ln2(x))
        return x

class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads, num_layers, feedforward_dim, context_length):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embedding_dim, num_heads, feedforward_dim) for _ in range(num_layers)])
        self.ln_final = nn.LayerNorm(embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)
    def forward(self, input_ids):
        B, T = input_ids.shape
        if T > self.context_length:
            raise ValueError("Input sequence exceeds context length.")
        positions = torch.arange(T, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.blocks(x)
        x = self.ln_final(x)
        return self.output(x)

# ---- Generation, loss, evaluation (class Cells 12A, 13A, 14G) ----
@torch.no_grad()
def generate_text(model, prompt, max_new_tokens=100, temperature=1.0):
    if temperature <= 0:
        raise ValueError("Temperature must be greater than zero.")
    was_training = model.training
    model.eval()
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        context = ids[:, -model.context_length:]          # the model only ever sees the last 32 characters
        logits = model(context)
        next_logits = logits[:, -1, :] / temperature
        probabilities = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probabilities, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    if was_training:
        model.train()
    return decode(ids[0].tolist())

def calculate_loss(logits, targets):
    B, T, V = logits.shape
    return F.cross_entropy(logits.reshape(B * T, V), targets.reshape(B * T))

@torch.no_grad()
def evaluate(model, source, batches=20):
    model.eval()
    losses = []
    for _ in range(batches):
        x, y = get_batch(source)
        losses.append(calculate_loss(model(x), y).item())
    model.train()
    return sum(losses) / len(losses)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

Matches Part 1 selected_slm.json: {'embedding_dim': 96, 'num_heads': 4, 'num_layers': 2, 'feedforward_dim': 384}
Characters in corpus: 90300 (Part 1: 90,300)
Vocabulary size: 27 -> '\n .abcdefghijklmnoprstuvwxy'
Letters the model has never seen: ['q', 'z']


## Load (or, the first time, retrain) the selected checkpoint
The Part 1 checkpoint `selected_slm.pt` is not in GitHub (the assignment keeps `*.pt` out of Git). If it is not in `slm_results/`, we retrain **Experiment 5 with the identical Part 1 code, settings and seed** and save it. Part 1's recorded results are printed next to ours as a reproducibility check. Small differences in the last decimals are normal when the same code runs on a different machine.

In [3]:
CKPT = OUT / "selected_slm.pt"
REPRO = OUT / "slm_repro_check.json"
PART1 = {"Total Parameters": 232155, "Final Training Loss": 0.1266, "Avg Loss Last 500 Steps": 0.1262,
         "Validation Loss": 0.1264, "Training Time (s)": 626.6}
CFG = dict(embedding_dim=EMBEDDING_DIM, num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
           feedforward_dim=FEEDFORWARD_DIM, context_length=CONTEXT_LENGTH)

def new_model():
    return TinyTransformerLM(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, num_heads=NUM_HEADS,
                             num_layers=NUM_LAYERS, feedforward_dim=FEEDFORWARD_DIM,
                             context_length=CONTEXT_LENGTH).to(DEVICE)

def train_selected_slm():
    # Identical to Part 1's run_experiment("5", ...): same seeds, same optimizer, same steps.
    torch.manual_seed(SEED)
    model = new_model()
    total_parameters = count_parameters(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    torch.manual_seed(SEED)
    model.train()
    loss_history = []
    start = time.perf_counter()
    for step in range(TRAINING_STEPS):
        xb, yb = get_batch(train_data)
        optimizer.zero_grad()
        loss = calculate_loss(model(xb), yb)
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
        if step % 5000 == 0 or step == TRAINING_STEPS - 1:
            print(f"  Step {step + 1:>5}/{TRAINING_STEPS} | Loss: {loss.item():.4f}")
    elapsed = time.perf_counter() - start
    torch.manual_seed(SEED)
    val_loss = evaluate(model, val_data)
    torch.save({"model_state": model.state_dict(), "vocab": chars, "config": CFG}, CKPT)
    record = {"Total Parameters": total_parameters, "Final Training Loss": round(loss_history[-1], 4),
              "Avg Loss Last 500 Steps": round(float(np.mean(loss_history[-500:])), 4),
              "Validation Loss": round(val_loss, 4), "Training Time (s)": round(elapsed, 1),
              "Machine": platform.platform(), "PyTorch": torch.__version__}
    json.dump(record, open(REPRO, "w"), indent=2)
    return model

if CKPT.exists():
    print("Loading existing checkpoint:", CKPT)
else:
    print("No checkpoint found: retraining Experiment 5 (30,000 steps, CPU) ...")
    train_selected_slm()

saved = torch.load(CKPT, map_location=DEVICE, weights_only=True)
assert saved["vocab"] == chars, "checkpoint vocabulary does not match the tokenizer"
slm = new_model()
slm.load_state_dict(saved["model_state"])
slm.eval()
for p in slm.parameters():
    p.requires_grad_(False)          # RAG must never update the weights
print("Trainable parameters of the loaded SLM (counted before freezing = architecture size):", count_parameters(new_model()))

if REPRO.exists():
    mine = json.load(open(REPRO))
    print()
    print(pd.DataFrame({"Part 1 (Colab)": PART1, "This run (retrained here)": {k: mine[k] for k in PART1}}))
    print("Retrained on:", mine["Machine"], "| PyTorch", mine["PyTorch"])
else:
    print("(Checkpoint was loaded, not retrained here, so there is no reproducibility record.)")

Loading existing checkpoint: slm_results/selected_slm.pt
Trainable parameters of the loaded SLM (counted before freezing = architecture size): 232155
(Checkpoint was loaded, not retrained here, so there is no reproducibility record.)


In [4]:
def fingerprint(model):
    # SHA-256 over every weight, so we can prove RAG did not change the model.
    h = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        h.update(name.encode()); h.update(tensor.detach().cpu().numpy().tobytes())
    return h.hexdigest()

FP_BEFORE = fingerprint(slm)
print("Weights fingerprint before RAG:", FP_BEFORE[:32], "...")

Weights fingerprint before RAG: e331f6e0eddc2ce8463a513412aa2dca ...


## Step 2.1: Create a new knowledge base
Eight short documents about a made-up *Unleased* sublease service (prices, refunds and perks are **invented for this demo**, not real policies). Each has a unique ID. The SLM never saw any of this: its training corpus is the 20 class sentences. The wording is restricted to the SLM's 27-character alphabet (no `q`, `z`, digits, capitals or punctuation other than `.`), which is a compatibility adjustment for the character-level tokenizer. No private or sensitive information is included.

In [5]:
KB = pd.DataFrame([
    {"doc_id": "doc01", "topic": "verification", "text": "a verified student badge needs a valid edu email address."},
    {"doc_id": "doc02", "topic": "pricing",      "text": "browsing listings is free and posting a listing costs ten dollars a month."},
    {"doc_id": "doc03", "topic": "refunds",      "text": "refunds are given within seven days if a listing is removed by mistake."},
    {"doc_id": "doc04", "topic": "visits",       "text": "a visit slot lasts one hour and each slot can be booked by one renter."},
    {"doc_id": "doc05", "topic": "privacy",      "text": "the street address stays hidden until the lister accepts a renter."},
    {"doc_id": "doc06", "topic": "membership",   "text": "premium members get early alerts for new listings near campus."},
    {"doc_id": "doc07", "topic": "favorites",    "text": "renters can save listings to a favorites list and compare them later."},
    {"doc_id": "doc08", "topic": "support",      "text": "support answers messages on weekdays from nine to five through the help page."},
])

# Compatibility with the SLM's character-level tokenizer, and separation from the training data
assert KB["doc_id"].is_unique
bad = {d: sorted(set(t) - set(stoi)) for d, t in zip(KB.doc_id, KB.text) if set(t) - set(stoi)}
assert not bad, f"characters the SLM cannot encode: {bad}"
assert not (set(KB.text) & CORPUS_SENTENCES), "a knowledge-base sentence is also in the training corpus"
kb_words = {w for t in KB.text for w in re.findall(r"[a-z]+", t)}
print(f"{len(KB)} documents, all encodable. {len(kb_words - CORPUS_WORDS)} of {len(kb_words)} distinct words "
      f"({len(kb_words - CORPUS_WORDS) / len(kb_words):.0%}) never appear in the SLM's training corpus.")
print("Characters per document:", KB.text.str.len().tolist(), "| model context window:", CONTEXT_LENGTH)
KB.to_csv("knowledge_base.csv", index=False)
KB

8 documents, all encodable. 66 of 75 distinct words (88%) never appear in the SLM's training corpus.
Characters per document: [57, 74, 71, 70, 66, 62, 69, 77] | model context window: 32


,doc_id,topic,text
0,doc01,verification,a verified student badge needs a valid edu email address.
1,doc02,pricing,browsing listings is free and posting a listing costs ten dollars a month.
2,doc03,refunds,refunds are given within seven days if a listing is removed by mistake.
3,doc04,visits,a visit slot lasts one hour and each slot can be booked by one renter.
4,doc05,privacy,the street address stays hidden until the lister accepts a renter.
5,doc06,membership,premium members get early alerts for new listings near campus.
6,doc07,favorites,renters can save listings to a favorites list and compare them later.
7,doc08,support,support answers messages on weekdays from nine to five through the help page.


## Step 2.2: Five questions, answered by the SLM without RAG (baseline)
Questions are lower case with no `?` (the SLM has no question mark). Each has a *gold* document and gold keywords, used only for scoring. Prompt format (all labels are in the SLM's alphabet): `ask. <question>` then a newline and `reply. `, and the model continues from there. Generation settings are Part 1's: temperature 0.7, 60 new characters, and the random seed is reset before every generation so any difference between two answers comes from the prompt alone.

In [6]:
QUESTIONS = pd.DataFrame([
    {"qid": "Q1", "question": "how much does it cost to post a listing",       "gold_doc": "doc02", "gold_keywords": ["ten", "dollars", "month"]},
    {"qid": "Q2", "question": "what do i need to get a verified badge",        "gold_doc": "doc01", "gold_keywords": ["edu", "email"]},
    {"qid": "Q3", "question": "when is the street address shown to a renter",  "gold_doc": "doc05", "gold_keywords": ["accepts", "lister"]},
    {"qid": "Q4", "question": "how long is a visit slot",                      "gold_doc": "doc04", "gold_keywords": ["one", "hour"]},
    {"qid": "Q5", "question": "what do premium members get",                   "gold_doc": "doc06", "gold_keywords": ["early", "alerts"]},
])
assert all(set(q) <= set(stoi) for q in QUESTIONS.question), "a question uses characters the SLM cannot encode"
GOLD = dict(zip(KB.doc_id, KB.text))
QUESTIONS["gold_answer"] = QUESTIONS.gold_doc.map(GOLD)
QUESTIONS.to_csv("questions.csv", index=False)

GEN_TEMPERATURE = 0.7      # Part 1 generation settings
MAX_NEW_TOKENS = 60

def answer(prompt):
    torch.manual_seed(SEED)
    full = generate_text(slm, prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=GEN_TEMPERATURE)
    return full[len(prompt):]

# ---- Response-quality rubric (1 to 10) ----
# We cannot hand-grade 45 answers fairly, so the score is a transparent formula. Read the answers too.
#   factual accuracy (50%): share of the gold keywords that appear in the response
#   relevance        (30%): share of the response's content words that come from the question or the gold document
#   support          (20%): share of the response's content words that appear in the retrieved documents
#   score = round(1 + 9 * (0.5 * accuracy + 0.3 * relevance + 0.2 * support)), so an answer that has nothing right scores 1.
STOP = set("the a an is are to of and in for if it do i what how when does get can be by on or that this with from at as each".split())
def words(s): return re.findall(r"[a-z]+", s.lower())
def content(s): return {w for w in words(s) if w not in STOP}

def score(response, row, retrieved_text):
    resp_words, resp_content = set(words(response)), content(response)
    accuracy = sum(k in resp_words for k in row.gold_keywords) / len(row.gold_keywords)
    relevant_pool = content(row.question) | content(row.gold_answer)
    relevance = len(resp_content & relevant_pool) / max(1, len(resp_content))
    support = len(resp_content & content(retrieved_text)) / max(1, len(resp_content))
    raw = 0.5 * accuracy + 0.3 * relevance + 0.2 * support
    return {"accuracy": round(accuracy, 2), "relevance": round(relevance, 2), "support": round(support, 2),
            "quality": int(max(1, min(10, round(1 + 9 * raw))))}

rows = []
for r in QUESTIONS.itertuples():
    prompt = f"ask. {r.question}\nreply. "
    resp = answer(prompt)
    rows.append({"Question": r.qid, "Question text": r.question, "SLM Response Without RAG": resp,
                 **score(resp, r, ""), "Gold answer": r.gold_answer})
baseline = pd.DataFrame(rows)
baseline.to_csv("baseline_no_rag.csv", index=False)
baseline[["Question", "Question text", "SLM Response Without RAG", "quality"]]

,Question,Question text,SLM Response Without RAG,quality
0,Q1,how much does it cost to post a listing,offers courses in informatics.\nstudents experiment with lang,1
1,Q2,what do i need to get a verified badge,the weigention generate text.\nthe model predicts characters,1
2,Q3,when is the street address shown to a renter,offers courses in informatics.\nstudents experiment with lang,1
3,Q4,how long is a visit slot,offers courses in informatics.\nstudents experiment with lang,1
4,Q5,what do premium members get,offers courses in informatics.\nstudents experiment with lang,1


**Is the rubric able to give a high score?** Every answer above scores 1 because every answer is an unrelated memorized sentence. To show that is the model's doing and not the scorer's, we score three *made-up* answers to Q1 (they are not from the SLM).

In [7]:
q1 = next(QUESTIONS.itertuples())
tests = {"perfect answer": "ten dollars a month.", "partly right": "it costs ten dollars.", "unrelated corpus sentence": "students build django applications."}
pd.DataFrame([{"made-up answer": a, **score(a, q1, GOLD["doc02"])} for a in tests.values()],
             index=list(tests)).rename(columns={"quality": "quality (1-10)"})

,made-up answer,accuracy,relevance,support,quality (1-10)
perfect answer,ten dollars a month.,1.00,1.0,1.0,10
partly right,it costs ten dollars.,0.67,1.0,1.0,8
unrelated corpus sentence,students build django applications.,0.00,0.0,0.0,1


## Step 2.3: Apply RAG to the same SLM
**Retrieval pipeline: the one from the Week 6 RAG notebook (class Cells 5A, 6A and 7), reused.** Each document (`title + text`) is converted to a TF-IDF vector with `TfidfVectorizer(stop_words="english", ngram_range=(1, 2))`, the question is vectorized the same way, `cosine_similarity` ranks the documents, and `retrieve(query, top_k, min_score=0.01)` returns the best matches (fewer than `top_k` if some documents have almost no word overlap with the question, exactly as in class). Only the generator changed: the class sent the prompt to an Ollama model, here it goes to our SLM.

**Compatibility adjustments for the SLM** (the assignment allows these):
- The class `build_rag_prompt()` is a long instruction paragraph with `DOCUMENTS:`, `QUESTION:` and `ANSWER:` labels and `[D1]`-style citations. The SLM cannot encode capitals, colons, digits or the letter `q`, and the whole paragraph would not fit in 32 characters, so the labels become `info.`, `ask.` and `reply.`, the instruction paragraph is dropped, and document IDs are not written into the prompt.
- Two layouts are tested with the same retrieved documents: **`standard`** (documents first, best first, then the question: the class order) and **`docs_last`** (question first, documents after it with the best one last, right before `reply.`, so the best document sits as close as possible to the model's 32-character window).

We also record the **window the SLM really sees** (the last 32 characters of the prompt) and how much of the retrieved text is inside it.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Class pipeline (Week 6 RAG notebook, Cells 5A and 6A), reading our knowledge base ---
documents = KB.rename(columns={"doc_id": "id", "topic": "title"}).to_dict("records")
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
document_matrix = vectorizer.fit_transform([d["title"] + " " + d["text"] for d in documents])   # documents -> searchable vectors
print("Document matrix shape:", document_matrix.shape, "| vocabulary features:", len(vectorizer.vocabulary_))

def retrieve(query, top_k=2, min_score=0.01):
    # Return top matching documents (best first) with cosine-similarity scores; same as the class function.
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_matrix)[0]
    indices = scores.argsort()[::-1][:top_k]
    return [{**documents[i], "score": float(scores[i])} for i in indices if scores[i] >= min_score]

# Retrieval quality by itself, before the SLM is involved
chk = []
for r in QUESTIONS.itertuples():
    ranked = [d["id"] for d in retrieve(r.question, len(documents), min_score=0.0)]
    chk.append({"Question": r.qid, "gold_doc": r.gold_doc, "top_1": ranked[0],
                "rank_of_gold": ranked.index(r.gold_doc) + 1,
                "docs_with_any_match": len(retrieve(r.question, len(documents))),
                "top_5": ", ".join(ranked[:5])})
retrieval_check = pd.DataFrame(chk)
retrieval_check.to_csv("retrieval_check.csv", index=False)
print("Gold document ranked first for", int((retrieval_check.rank_of_gold == 1).sum()), "of", len(retrieval_check), "questions")
retrieval_check

Document matrix shape: (8, 115) | vocabulary features: 115
Gold document ranked first for 5 of 5 questions


,Question,gold_doc,top_1,rank_of_gold,docs_with_any_match,top_5
0,Q1,doc02,doc02,1,2,"doc02, doc03, doc08, doc07, doc06"
1,Q2,doc01,doc01,1,1,"doc01, doc08, doc07, doc06, doc05"
2,Q3,doc05,doc05,1,3,"doc05, doc04, doc01, doc08, doc07"
3,Q4,doc04,doc04,1,1,"doc04, doc08, doc07, doc06, doc05"
4,Q5,doc06,doc06,1,1,"doc06, doc08, doc07, doc05, doc04"


In [9]:
def assemble(segments):
    # Joins (kind, text) pieces into a prompt and remembers where each document sits.
    prompt, spans = "", []
    for kind, s in segments:
        start = len(prompt); prompt += s
        if kind.startswith("doc"):
            spans.append((start, len(prompt)))
    return prompt, spans

def build_prompt(question, docs, layout):
    # docs: retrieved document dicts, best first. No documents -> the plain no-RAG prompt.
    if not docs:
        return assemble([("t", f"ask. {question}\nreply. ")])
    if layout == "standard":
        seg = [("t", "info. ")]
        for d in docs:
            seg += [(d["id"], d["text"]), ("t", "\n")]
        seg += [("t", f"ask. {question}\nreply. ")]
    else:  # docs_last: best document ends up directly before "reply. "
        seg = [("t", f"ask. {question}\ninfo. ")]
        for i, d in enumerate(reversed(docs)):
            seg += ([("t", "\n")] if i else []) + [(d["id"], d["text"])]
        seg += [("t", "\nreply. ")]
    return assemble(seg)

def run_rag(layout, ks=(1, 2, 3, 5)):
    out = []
    for r in QUESTIONS.itertuples():
        base_resp = baseline.loc[baseline.Question == r.qid, "SLM Response Without RAG"].iloc[0]
        for k in ks:
            docs = retrieve(r.question, k)
            ids = [d["id"] for d in docs]
            prompt, spans = build_prompt(r.question, docs, layout)
            assert set(prompt) <= set(stoi), "prompt contains characters the SLM cannot encode"
            win_start = max(0, len(prompt) - CONTEXT_LENGTH)
            doc_chars = sum(e - s for s, e in spans)
            seen = sum(max(0, e - max(s, win_start)) for s, e in spans)
            resp = answer(prompt)
            out.append({"Question": r.qid, "Top-K": k, "Retrieved Documents": ", ".join(ids) or "(none)",
                        "Gold doc retrieved": r.gold_doc in ids,
                        "Prompt chars": len(prompt), "Window the SLM sees": prompt[win_start:],
                        "Retrieved text visible": f"{seen}/{doc_chars} chars",
                        "Generated Response": resp, "Same as no-RAG answer": resp == base_resp,
                        **score(resp, r, " ".join(d["text"] for d in docs))})
            out[-1]["Response Quality (1-10)"] = out[-1].pop("quality")
    return pd.DataFrame(out)

rag_standard = run_rag("standard")
rag_standard.to_csv("rag_results.csv", index=False)
rag_standard[["Question", "Top-K", "Retrieved Documents", "Retrieved text visible", "Generated Response", "Response Quality (1-10)"]]

,Question,Top-K,Retrieved Documents,Retrieved text visible,Generated Response,Response Quality (1-10)
0,Q1,1,doc02,0/74 chars,offers courses in informatics.\nstudents experiment with lang,1
1,Q1,2,"doc02, doc03",0/145 chars,offers courses in informatics.\nstudents experiment with lang,1
2,Q1,3,"doc02, doc03",0/145 chars,offers courses in informatics.\nstudents experiment with lang,1
3,Q1,5,"doc02, doc03",0/145 chars,offers courses in informatics.\nstudents experiment with lang,1
4,Q2,1,doc01,0/57 chars,the weigention generate text.\nthe model predicts characters,1
5,Q2,2,doc01,0/57 chars,the weigention generate text.\nthe model predicts characters,1
6,Q2,3,doc01,0/57 chars,the weigention generate text.\nthe model predicts characters,1
7,Q2,5,doc01,0/57 chars,the weigention generate text.\nthe model predicts characters,1
8,Q3,1,doc05,0/66 chars,offers courses in informatics.\nstudents experiment with lang,1
9,Q3,2,"doc05, doc04",0/136 chars,offers courses in informatics.\nstudents experiment with lang,1


### The same experiment with the `docs_last` layout (documents placed right before `reply.`)

In [10]:
rag_adjusted = run_rag("docs_last")
rag_adjusted.to_csv("rag_results_docs_last_layout.csv", index=False)
rag_adjusted[["Question", "Top-K", "Retrieved Documents", "Window the SLM sees", "Retrieved text visible", "Generated Response", "Response Quality (1-10)"]]

,Question,Top-K,Retrieved Documents,Window the SLM sees,Retrieved text visible,Generated Response,Response Quality (1-10)
0,Q1,1,doc02,sts ten dollars a month.\nreply.,24/74 chars,learn python programming.\nstudents develop artificial intell,1
1,Q1,2,"doc02, doc03",sts ten dollars a month.\nreply.,24/145 chars,learn python programming.\nstudents develop artificial intell,1
2,Q1,3,"doc02, doc03",sts ten dollars a month.\nreply.,24/145 chars,learn python programming.\nstudents develop artificial intell,1
3,Q1,5,"doc02, doc03",sts ten dollars a month.\nreply.,24/145 chars,learn python programming.\nstudents develop artificial intell,1
4,Q2,1,doc01,valid edu email address.\nreply.,24/57 chars,offers courses in informatics.\nstudents experiment with lang,1
5,Q2,2,doc01,valid edu email address.\nreply.,24/57 chars,offers courses in informatics.\nstudents experiment with lang,1
6,Q2,3,doc01,valid edu email address.\nreply.,24/57 chars,offers courses in informatics.\nstudents experiment with lang,1
7,Q2,5,doc01,valid edu email address.\nreply.,24/57 chars,offers courses in informatics.\nstudents experiment with lang,1
8,Q3,1,doc05,lister accepts a renter.\nreply.,24/66 chars,offers courses in informatics.\nstudents experiment with lang,1
9,Q3,2,"doc05, doc04",lister accepts a renter.\nreply.,24/136 chars,offers courses in informatics.\nstudents experiment with lang,1


### Summary numbers used in the analysis

In [11]:
summary = pd.concat([
    pd.DataFrame({"Setting": ["No RAG"], "Mean quality": [baseline.quality.mean()],
                  "Mean accuracy": [baseline.accuracy.mean()], "Answers equal to no-RAG": [5]}),
    *[pd.DataFrame({"Setting": [f"{name}, Top-{k}" for k in (1, 2, 3, 5)],
                    "Mean quality": [df[df["Top-K"] == k]["Response Quality (1-10)"].mean() for k in (1, 2, 3, 5)],
                    "Mean accuracy": [df[df["Top-K"] == k]["accuracy"].mean() for k in (1, 2, 3, 5)],
                    "Answers equal to no-RAG": [int(df[df["Top-K"] == k]["Same as no-RAG answer"].sum()) for k in (1, 2, 3, 5)]})
      for name, df in (("standard", rag_standard), ("docs_last", rag_adjusted))]
], ignore_index=True).round(2)
summary["Out of"] = [5] * len(summary)
summary.to_csv("rag_summary.csv", index=False)
print(summary.to_string(index=False))

print()
for name, df in (("standard", rag_standard), ("docs_last", rag_adjusted)):
    distinct = df.groupby("Question")["Generated Response"].nunique().to_dict()
    print(f"{name}: distinct responses across Top-1/2/3/5 per question ->", distinct)
    print(f"{name}: gold document retrieved in", int(df["Gold doc retrieved"].sum()), "of", len(df), "runs;",
          "avg share of retrieved characters inside the 32-char window:",
          round(df["Retrieved text visible"].map(lambda s: int(s.split('/')[0]) / max(1, int(s.split('/')[1].split()[0]))).mean(), 2))

# Is any 'new' word (not in the training corpus) ever produced in the answers?
new_words = {w for df in (rag_standard, rag_adjusted) for resp in df["Generated Response"] for w in words(resp)} - CORPUS_WORDS
print("\nWords in answers that are not corpus words (made-up or copied from the knowledge base):", sorted(new_words)[:25])
print("Of those, words that come from the knowledge base:", sorted(new_words & kb_words))

         Setting  Mean quality  Mean accuracy  Answers equal to no-RAG  Out of
          No RAG           1.0            0.0                        5       5
 standard, Top-1           1.0            0.0                        5       5
 standard, Top-2           1.0            0.0                        5       5
 standard, Top-3           1.0            0.0                        5       5
 standard, Top-5           1.0            0.0                        5       5
docs_last, Top-1           1.0            0.0                        3       5
docs_last, Top-2           1.0            0.0                        3       5
docs_last, Top-3           1.0            0.0                        3       5
docs_last, Top-5           1.0            0.0                        3       5

standard: distinct responses across Top-1/2/3/5 per question -> {'Q1': 1, 'Q2': 1, 'Q3': 1, 'Q4': 1, 'Q5': 1}
standard: gold document retrieved in 20 of 20 runs; avg share of retrieved characters inside the 32

## Proof that RAG did not change the model

In [12]:
FP_AFTER = fingerprint(slm)
print("before:", FP_BEFORE)
print("after :", FP_AFTER)
assert FP_BEFORE == FP_AFTER
print("Identical: the weights were never updated; every difference above comes from the prompt alone.")

before: e331f6e0eddc2ce8463a513412aa2dca8f7584badc0d1ec9d27237cd00cd62d6
after : e331f6e0eddc2ce8463a513412aa2dca8f7584badc0d1ec9d27237cd00cd62d6
Identical: the weights were never updated; every difference above comes from the prompt alone.


## Step 2.4: Analysis

**Setup recap:** SLM = Part 1's own `selected_slm.pt`, Experiment 5 (232,155 parameters), loaded without any weight update; the checksum before and after RAG is identical. 8 knowledge-base documents, 5 questions, retrieval Top-1/2/3/5, two prompt layouts (`standard` and `docs_last`). The class retriever (word TF-IDF + cosine similarity) ranked the right document first for **5 of 5** questions and had it inside the retrieved set in **20 of 20** runs, so retrieval worked; the problem is what the SLM can do with the result. Because the class `retrieve()` drops documents with almost no word overlap (`min_score=0.01`), fewer than K documents came back for some questions: Q2, Q4 and Q5 retrieved a single document at every K, Q1 at most 2 and Q3 at most 3.

| Setting | Mean quality (1-10) | Mean factual accuracy | Answers identical to no-RAG (of 5) |
|---|---|---|---|
| No RAG | 1.0 | 0.0 | n/a |
| standard, Top-1 / 2 / 3 / 5 | 1.0 each | 0.0 | 5 / 5 / 5 / 5 |
| docs_last, Top-1 / 2 / 3 / 5 | 1.0 each | 0.0 | 3 / 3 / 3 / 3 |

**1. Did providing retrieved information change the SLM's responses?**
Barely, and never in a useful way. With the standard layout 0 of 20 answers changed, because the 32-character window the model sees is exactly the same as in the no-RAG prompt; with `docs_last` 2 of 5 questions (Q1 and Q2) got a different answer, but it was just another memorized corpus sentence (Q1 went from "offers courses in informatics..." to "learn python programming. students develop artificial intell...") produced because different characters were now in the window.

**2. Did RAG improve factual accuracy?**
No. Accuracy was 0.0 in the baseline and in all 40 RAG runs, none of the 45 answers contained a gold keyword or any word that exists only in the knowledge base (the only non-corpus words were the cut-off fragments "intell" and "lang" and one garbled word, "weigention", which the SLM also produces for Q2 with no retrieval at all), and every answer scored 1 out of 10, even though the right document was retrieved every time.

**3. Did retrieving more documents always improve the responses?**
No: Top-1, 2, 3 and 5 gave exactly the same answer for every question in both layouts, and mean quality stayed at 1.0. For Q2, Q4 and Q5 the retriever found only one matching document, so a larger K changed nothing there; for Q1 and Q3, where extra documents were retrieved, they only made the prompt longer (109 to 133 characters at Top-1, up to 259 at larger K) and diluted the useful text: with `docs_last` the visible part of the retrieved text fell from 24 of 66 characters at Top-1 to 24 of 193 for Q3 at Top-3 and Top-5.

**4. How did the SLM's context-length limitation affect the results?**
The model only ever sees the last 32 characters, while one document alone is 57 to 77 characters and the longest prompt is 259 characters. In the standard layout 0% of the retrieved text was inside the window (the model saw only the end of the question and "reply."), and in the `docs_last` layout the best case was 24 characters, the tail of one document (for example "lister accepts a renter."), so the full fact was never visible.

**5. Why might RAG behave differently with a small character-level model than with a larger pretrained model?**
This SLM was only trained to continue 20 memorized sentences, so it never learned to read a prompt and copy or reason over facts in it (in-context learning), 88% of the knowledge-base words were new to it, and one fact costs 60+ characters of its 32-character window. A large pretrained model has thousands of tokens of context and has learned from huge, varied text to use whatever information appears in the prompt, which is why RAG works there but only produces memorized corpus sentences here.

**Honest limits of this experiment**
- The quality score is a transparent formula (50% keyword accuracy, 30% relevance, 20% support by retrieved text), not a human grade; the rubric self-test above shows it can award 10, and every answer was also read: all are unrelated corpus sentences.
- The checkpoint is Part 1's own `selected_slm.pt` (it is kept out of GitHub, as the assignment asks; put it in `slm_results/` and Run all to reproduce these results). As a cross-check, a copy we retrained from scratch with Part 1's code and seed on a different machine (232,155 parameters; last-500-step loss 0.1260 vs 0.1262) gave the same pattern: every answer scored 1, no standard-layout answer changed, and 2 of 5 `docs_last` answers changed. Only the exact memorized sentences differed.
- Retrieval is the Week 6 class pipeline (word-level TF-IDF, cosine similarity, `min_score=0.01`), so it matches words, not meaning; the class notebook's optional neural-embedding retrieval was not used.
